# iprPy dislocation_dipole calculation

In [1]:
# Standard library imports
import datetime
from copy import deepcopy

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-06-26 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('dislocation_dipole')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# dislocation_dipole calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

## Introduction

The dislocation_dipole calculation style calculation creates a small-cell dislocation dipole configuration consisting of two dislocations of opposite sign.  This type of cell allows for all three dimensions to be periodic and stabilizes the dislocation core positions by applying a counteracting strain.  The resulting configuration is consistent with a periodic 2D array of dislocations with alternating burgers vector signs.

### Version notes

- 2024-12-19: Calculation added
- v0.12.0: Method updated to support the LAMMPS library interface.
  
### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)
- This calculation is best used with dislocation cores that remain compact and do not split or spread along the slip plane as the cores should not overlap or interfere with each other.
- The allowed size dimensions is a little unintuitive due to being dependent on the crystal symmetry. Trying guess values on a test system is usually adequate for finding good values.
- The calculation allows for the system to be relaxed either using only static energy/force minimizations or with molecular dynamic steps followed by a minimization.  Only performing a static relaxation is considerably faster than performing a dynamic relaxation, but the dynamic relaxation is more likely to result in a better final dislocation structure. If a dynamic relaxation is performed, the temperature should be kept low to prevent the dislocations from slipping and potentially annihilating each other.


## Method and Theory

### Dislocation construction

The construction of dislocations using the Volterra dislocation solutions corresponds to what is described in the dislocation_monopole documentation.  Constructing a dipole configuration builds upon this by:

1. Obtaining two solutions of the Volterra dislocation, one with a positive Burgers vector and one with a negative Burgers vector that are located equidistant between replicas along the slip plane.  The displacement fields for the two solutions are added together to produce an approximate displacement field for the dipole system.
2. The approximate dipole solution is used to compute the displacement that should act on the cell from a number of neighboring periodic cells.  These neighbor cell solutions are added to the approximate solution.
3. A linear displacement is added on to correct for the error in using a limited number of periodic cells in step 2.  This displacement is identified as ensuring atomic compatibility across the periodic boundaries.
4. A strain is applied to the system to counteract the elastic attraction between the two dislocations.



## 2. Display the underlying code

This section displays the underlying code used when calculation.calc() is called in Section 4. It is provided here allowing any users to see and understand how the calculation works.

Feel free to modify and test the functions for yourself!  You can do this by
1. Copy the Python code displayed here into a "Code" cell and run it.
2. In Section 2.2, set "savefiles = True" and run the cell to save any supporting non-python files to the working directory.
3. In Section 4, call the primary calculation function directly rather than using calculation.calc().

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "dislocation_dipole.py"

# Python script created by Lucas Hale

# Standard library imports
from pathlib import Path
from typing import Optional, Union

# http://www.numpy.org/
import numpy as np 

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential, unitfloat, millerindices, lammps
from atomman.lammps import LAMMPS, LAMMPSobj

def dislocation_dipole(lammps_command: Union[str, LAMMPSobj],
                       ucell: am.System,
                       potential: lammpspotential,
                       C: am.ElasticConstants,
                       burgers: millerindices,
                       ξ_uvw: millerindices,
                       slip_hkl: millerindices,
                       mpi_command: Optional[str] = None,
                       m: Union[list, np.ndarray] = [0,1,0],
                       n: Union[list, np.ndarray] = [0,0,1],
                       conventional_setting: str = 'p',
                       sizemults = None,
                       shift: Union[list, np.ndarray, None] = None,
                       shiftscale: bool = False,
                       shiftindex: Optional[int] = None,
                       tol: float = 1e-8,
                       etol: float = 0.0,
                       ftol: unitfloat = 0.0,
                       maxiter: int = 10000,
                       maxeval: int = 100000,
                       dmax: unitfloat = '0.01 angstrom',
                       annealtemp: float = 0.0,
                       annealsteps: Optional[int] = None,
                       randomseed: Optional[int] = None,
                       usefiles: bool = False) -> dict:
    """
    Creates and relaxes a dislocation dipole system.
    
    Parameters
    ----------
    lammps_command :str
        Command for running LAMMPS.
    ucell : atomman.System
        The unit cell to use as the seed for generating the dislocation
        monopole system.
    potential : PotentialLAMMPS or PotentialLAMMPSKIM
        The LAMMPS implemented potential to use.
    C : atomman.ElasticConstants
        The elastic constants associated with the bulk crystal structure
        for ucell.
    burgers : array-like object
        The dislocation's Burgers vector given as a Miller or
        Miller-Bravais vector relative to ucell.
    ξ_uvw : array-like object
        The dislocation's line direction given as a Miller or
        Miller-Bravais vector relative to ucell.
    slip_hkl : array-like object
        The dislocation's slip plane given as a Miller or Miller-Bravais
        plane relative to ucell.
    mpi_command : str or None, optional
        The MPI command for running LAMMPS in parallel.  If not given, LAMMPS
        will run serially.
    m : array-like object, optional
        The m unit vector for the dislocation solution.  m, n, and ξ
        (dislocation line) should be right-hand orthogonal.  Default value
        is [0,1,0] (y-axis).
    n : array-like object, optional
        The n unit vector for the dislocation solution.  m, n, and ξ
        (dislocation line) should be right-hand orthogonal.  Default value
        is [0,0,1] (z-axis). n is normal to the dislocation slip plane.
    conventional_setting : str, optional
        Indicates the space lattice setting of the given unit cell, i.e.
        'p' for primitive, 'i' for body-centered, 'f' for face-centered,
        'a', 'b', or 'c' for side-centered and 't1', or 't2' for trigonal
        in a hexagonal setting.  Setting this with the appropriate
        conventional unit cell allows for identifying lattice vectors that
        are not integers with respect to the conventional unit cell.  This
        also creates the rotated cell from a compatible primitive cell,
        thereby the final dislocation configurations can be smaller than
        possible solely from the conventional unit cell.
    sizemults : tuple, optional
        The size multipliers to use when generating the system.  Values 

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [ ]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [8]:
lammps_command = 'lmp_serial'
mpi_command = None
#mpi_command = 'mpiexec -localonly 4'

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 23 Jun 2022


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [9]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. Initial unit cell system

- __ucell__ is an atomman.System representing a fundamental unit cell of the system (required).  Here, this is generated by loading the relaxed fcc crystal for the potential from the database.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [10]:
# Create ucell by loading prototype record
ucell = am.load('crystal', potential=potential, family='A1--Cu--fcc')

print(ucell)

Multiple matching record retrieved from local
#  family               symbols  alat    Ecoh    method  standing
 1 A1--Cu--fcc          Ni        3.5200 -4.4500 dynamic good
 2 A1--Cu--fcc          Ni        7.3760  0.0119 dynamic good


Please select one: 1


avect =  [ 3.520,  0.000,  0.000]
bvect =  [ 0.000,  3.520,  0.000]
cvect =  [ 0.000,  0.000,  3.520]
origin = [ 0.000,  0.000,  0.000]
natoms = 4
natypes = 1
symbols = ('Ni',)
pbc = [ True  True  True]
per-atom properties = ['atype', 'pos']
     id |   atype |  pos[0] |  pos[1] |  pos[2]
      0 |       1 |   0.000 |   0.000 |   0.000
      1 |       1 |   0.000 |   1.760 |   1.760
      2 |       1 |   1.760 |   0.000 |   1.760
      3 |       1 |   1.760 |   1.760 |   0.000


### 3.4. Elastic constants

- __C_dict__ is a dictionary containing the unique elastic constants for the potential and crystal structure defined above. 
- __C__ is an atomman.ElasticConstants object built from C_dict.

In [11]:
C_dict = {}
C_dict['C11'] = uc.set_in_units(247.86, 'GPa')
C_dict['C12'] = uc.set_in_units(147.83, 'GPa')
C_dict['C44'] = uc.set_in_units(124.84, 'GPa')

# -------------- Derived parameters -------------- #
# Build ElasticConstants object from C_dict terms
C = am.ElasticConstants(**C_dict)

### 3.5. Defect parameters

- __burgers__ is the crystallographic Miller Burgers vector for the dislocation. 
- __ξ_uvw__ is the Miller \[uvw\] line vector direction for the dislocation.  The angle between burgers and ξ_uvw determines the dislocation's character
- __slip_hkl__ is the Miller (hkl) slip plane for the dislocation.
- __m__ is the Cartesian vector of the final system that the dislocation solution's m vector (in-plane, perpendicular to ξ) should align with.  Limited to being parallel to one of the three Cartesian axes.  
- __n__ is the Cartesian vector of the final system that the dislocation solution's n vector (slip plane normal) should align with.  Limited to being parallel to one of the three Cartesian axes. 
- __shift__ is a rigid body shift to apply to the atoms in the system. This controls how the atomic positions align with the ideal position of the dislocation core, which is at coordinates (0,0) for the two Cartesian axes aligned with m and n.
- __shiftscale__ allows for shift to be defined relative to the cell created by rotating ucell to coincide with the dislocation solution orientation.  This is useful as it allows for shift values to be defined relative to the defect type and crystal prototype rather than on a per-crystal basis. 
- __shiftindex__ alternate to specifying shift values, the shiftindex allows for one of the identified suggested shift values to be used that will position the slip plane halfway between two planes of atoms.  Note that shiftindex values only shift atoms in the slip plane normal direction and may not be the ideal positions for some dislocation cores.

In [12]:
# fcc a/2 <110>{111} dislocations
burgers = 0.5 * np.array([ 1., -1., 0.])
slip_hkl = np.array([1, 1, 1])

# Line direction determines dislocation character
ξ_uvw = np.array([ 1, 1,-2]) # 90 degree edge
#ξ_uvw = np.array([ 1, 0,-1]) # 60 degree mixed
#ξ_uvw = np.array([ 1,-2, 1]) # 30 degree mixed
#ξ_uvw = np.array([ 1,-1, 0]) # 0 degree screw

# Best choice for m + n as it works for non-cubic systems
m = [0,1,0]
n = [0,0,1]

# Specify shift or shiftindex
shift = None
shiftscale = True
shiftindex = 0

### 3.6. Calculation-specific parameters

- __annealtemperature__ is the temperature at which to relax (anneal) the dislocation system. If annealtemperature is 0.0, then only a static relaxation will be performed. Default value is 0.0.
- __annealsteps__ is the number of nvt iteration steps to perform at the given temperature.  Default value is 0 if annealtemperature is zero, 10000 otherwise.
- __randomseed__ allows for the random seed used in generating initial atomic velocities for a dynamic relaxation to be specified. This is an integer between 1 and 900000000. Default value is None, which will randomly pick a number in that range.
- __energytolerance__ is the energy tolerance to use during the minimizations. This is unitless.
- __forcetolerance__ is the force tolerance to use during the minimizations. This is in energy/length units.
- __maxiterations__ is the maximum number of minimization iterations to use.
- __maxevaluations__ is the maximum number of minimization evaluations to use.
- __maxatommotion__ is the largest distance that an atom is allowed to move during a minimization iteration. This is in length units. 

In [13]:
# Specify MD anneal parameters
annealtemperature = 50.0
annealsteps = 10000
randomseed = None

# Specify minimization parameters
energytolerance = 0.0
forcetolerance = uc.set_in_units(1e-6, 'eV/angstrom')
maxiterations = 10000
maxevaluations = 100000
maxatommotion = uc.set_in_units(0.01, 'angstrom')

### 3.7. System modifications

- __sizemults__ list of three sets of two integers specifying how many times the ucell vectors of $a$, $b$ and $c$ are replicated in positive and negative directions when creating system.

In [14]:
sizemults = [1, 40, 40]

## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [ ]:
# What is calculation.calc an alias of?
calculation.calc.__module__

In [15]:
results_dict = calculation.calc(lammps_command, ucell, potential, C,
                                burgers, ξ_uvw, slip_hkl,
                                mpi_command = mpi_command,
                                m = m,
                                n = n,
                                shift = shift,
                                shiftscale = shiftscale,
                                shiftindex = shiftindex,
                                sizemults = sizemults,
                                etol = energytolerance,
                                ftol = forcetolerance,
                                maxiter = maxiterations,
                                maxeval = maxevaluations,
                                dmax = maxatommotion,
                                annealtemp = annealtemperature, 
                                annealsteps = annealsteps,
                                randomseed = randomseed)
print(results_dict.keys())

dict_keys(['dumpfile_base', 'symbols_base', 'dislocation', 'dumpfile_disl', 'symbols_disl', 'E_total_disl'])


### 4.2. Report results

Values returned in the results_dict:

- **'dumpfile_base'** (*str*) - The filename of the LAMMPS dump file
  for the relaxed base system.
- **'symbols_base'** (*list of str*) - The list of element-model
  symbols for the Potential that correspond to the base system's
  atypes.
- **'dumpfile_disl'** (*str*) - The filename of the LAMMPS dump file
  for the relaxed dislocation monopole system.
- **'symbols_disl'** (*list of str*) - The list of element-model
  symbols for the Potential that correspond to the dislocation
  monopole system's atypes.
- **'dislocation'** (*atomman.defect.Dislocation*) - The Dislocation
  object used to generate the monopole system.
- **'E_total_disl'** (*float*) - The total potential energy of the
  dislocation monopole system.

In [16]:
length_unit = 'angstrom'
energy_unit = 'eV'
pressure_unit = 'GPa'
energy_per_length_unit = energy_unit+'/'+length_unit

In [17]:
print('pre-ln factor (alpha) =', uc.get_in_units(results_dict['dislocation'].dislsol.preln, energy_per_length_unit), energy_per_length_unit)
print('K_tensor =')
print(uc.get_in_units(results_dict['dislocation'].dislsol.K_tensor, pressure_unit), pressure_unit)

pre-ln factor (alpha) = 0.3811520216962063 eV/angstrom
K_tensor =
[[ 81.07544348   0.         -12.76027037]
 [  0.         123.86918865   0.        ]
 [-12.76027037   0.         127.31323026]] GPa


In [18]:
print('Perfect system is saved as', results_dict['dumpfile_base'])
print('with symbols', results_dict['symbols_base'])
print('Defect system is saved as',  results_dict['dumpfile_disl'])
print('with symbols', results_dict['symbols_disl'])

Perfect system is saved as base.dump
with symbols ('Ni',)
Defect system is saved as disl.dump
with symbols ('Ni', 'Ni')
